In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='tqdm')

import sys
sys.path.append('..')  # Go up one level to project root

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from datasets import load_dataset
import pandas as pd
import time
import os

# Import from custom modules
from src.preprocess import build_vocab, SentimentDataset, collate_fn
from src.mlp_model import MLPClassifier
from src.train import train_epoch
from src.evaluate import get_predictions, calculate_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Data Loading & Preprocessing ---
print("Loading IMDB dataset...")
dataset = load_dataset("imdb")
train_texts = dataset['train']['text']
train_labels = dataset['train']['label']

print(f"Dataset size: {len(train_texts)} reviews")

def count_parameters(model):
    """Calculates the total number of trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Build vocabulary
VOCAB_SIZE = 20000 
vocab = build_vocab(train_texts, max_size=VOCAB_SIZE)
print(f"Vocabulary size: {len(vocab)}")

# Create full dataset
full_dataset = SentimentDataset(train_texts, train_labels, vocab, max_len=256)

# Split into 70% train, 10% val, 20% test
train_size = int(0.7 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

print(f"Train set: {len(train_dataset)}, Val set: {len(val_dataset)}, Test set: {len(test_dataset)}")

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)


# Ensure checkpoints directory exists
os.makedirs("../checkpoints", exist_ok=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using device: cuda
Loading IMDB dataset...
Dataset size: 25000 reviews
Vocabulary size: 20000
Train set: 17500, Val set: 2500, Test set: 5000


In [13]:

# ============================================================================
# EXPERIMENT 1: Network Depth (1 vs 2 vs 3 hidden layers)
# ============================================================================
print("\n" + "="*70)
print("EXPERIMENT 1: Network Depth Comparison")
print("="*70)

exp1_configs = [
    {"name": "1 hidden layer", "hidden_dims": [256], "embed_dim": 128},
    {"name": "2 hidden layers", "hidden_dims": [128, 128], "embed_dim": 128},
    {"name": "3 hidden layers", "hidden_dims": [85, 85, 85], "embed_dim": 128},
]

exp1_results = []

for cfg in exp1_configs:
    print(f"\nTraining: {cfg['name']} (embed_dim={cfg['embed_dim']}, hidden_dims={cfg['hidden_dims']})")
    
    model = MLPClassifier(len(vocab), cfg['embed_dim'], cfg['hidden_dims'], 2, dropout=0.3).to(device)
    num_params = count_parameters(model)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_acc = 0.0
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    
    start_time = time.time()
    
    for epoch in range(10):
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, is_rnn=False)
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        # Validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                val_loss += criterion(outputs, labels).item() * inputs.size(0)
        val_loss /= len(val_dataset)
        val_losses.append(val_loss)
        
        # Get validation accuracy
        val_true, val_pred = get_predictions(model, val_loader, device, is_rnn=False)
        val_acc = calculate_metrics(val_true, val_pred)['Accuracy']
        val_accs.append(val_acc)
        
        print(f"  Epoch {epoch+1}/10 | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"../checkpoints/mlp_exp1_{cfg['name'].replace(' ', '_')}.pt")
    
    end_time = time.time()
    epoch_time = (end_time - start_time) / 10
    
    # Evaluate on test set
    model.load_state_dict(torch.load(f"../checkpoints/mlp_exp1_{cfg['name'].replace(' ', '_')}.pt"))
    test_true, test_pred = get_predictions(model, test_loader, device, is_rnn=False)
    test_metrics = calculate_metrics(test_true, test_pred)
    
    result = {
        "Config": cfg['name'],
        "Accuracy": test_metrics['Accuracy'],
        "Precision": test_metrics['Precision'],
        "Recall": test_metrics['Recall'],
        "F1": test_metrics['F1 Score'],
        "Params": num_params,
        "Time/epoch": epoch_time
    }
    exp1_results.append(result)
    print(f"  Test Accuracy: {test_metrics['Accuracy']:.4f}")

exp1_df = pd.DataFrame(exp1_results)
print("\nExperiment 1 Results:")
print(exp1_df.to_string(index=False))


EXPERIMENT 1: Network Depth Comparison

Training: 1 hidden layer (embed_dim=128, hidden_dims=[256])
  Epoch 1/10 | Train Loss: 0.5373 | Train Acc: 0.7258 | Val Acc: 0.8060
  Epoch 2/10 | Train Loss: 0.3373 | Train Acc: 0.8530 | Val Acc: 0.8384
  Epoch 3/10 | Train Loss: 0.2505 | Train Acc: 0.8965 | Val Acc: 0.8560
  Epoch 4/10 | Train Loss: 0.1926 | Train Acc: 0.9259 | Val Acc: 0.8656
  Epoch 5/10 | Train Loss: 0.1432 | Train Acc: 0.9498 | Val Acc: 0.8684
  Epoch 6/10 | Train Loss: 0.1034 | Train Acc: 0.9664 | Val Acc: 0.8660
  Epoch 7/10 | Train Loss: 0.0697 | Train Acc: 0.9807 | Val Acc: 0.8616
  Epoch 8/10 | Train Loss: 0.0464 | Train Acc: 0.9869 | Val Acc: 0.8540
  Epoch 9/10 | Train Loss: 0.0280 | Train Acc: 0.9946 | Val Acc: 0.8544
  Epoch 10/10 | Train Loss: 0.0162 | Train Acc: 0.9974 | Val Acc: 0.8508
  Test Accuracy: 0.8640

Training: 2 hidden layers (embed_dim=128, hidden_dims=[128, 128])
  Epoch 1/10 | Train Loss: 0.5344 | Train Acc: 0.7235 | Val Acc: 0.8032
  Epoch 2/10 | 

In [14]:
# ============================================================================
# EXPERIMENT 2: Embedding Dimension (64, 128, 256)
# Using best architecture from Experiment 1
# ============================================================================
print("\n" + "="*70)
print("EXPERIMENT 2: Embedding Dimension (using best depth from Exp1)")
print("="*70)

best_depth = exp1_df.loc[exp1_df['Accuracy'].idxmax(), 'Config']
best_hidden_dims = exp1_configs[[c['name'] for c in exp1_configs].index(best_depth)]['hidden_dims']

exp2_configs = [
    {"name": "embed_64", "embed_dim": 64, "hidden_dims": best_hidden_dims},
    {"name": "embed_128", "embed_dim": 128, "hidden_dims": best_hidden_dims},
    {"name": "embed_256", "embed_dim": 256, "hidden_dims": best_hidden_dims},
]

exp2_results = []

for cfg in exp2_configs:
    print(f"\nTraining: {cfg['name']} (embed_dim={cfg['embed_dim']}, hidden_dims={cfg['hidden_dims']})")
    
    model = MLPClassifier(len(vocab), cfg['embed_dim'], cfg['hidden_dims'], 2, dropout=0.3).to(device)
    num_params = count_parameters(model)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_acc = 0.0
    
    start_time = time.time()
    
    for epoch in range(10):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, is_rnn=False)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                val_loss += criterion(outputs, labels).item() * inputs.size(0)
        val_loss /= len(val_dataset)
        
        val_true, val_pred = get_predictions(model, val_loader, device, is_rnn=False)
        val_acc = calculate_metrics(val_true, val_pred)['Accuracy']
        
        print(f"  Epoch {epoch+1}/10 | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"../checkpoints/mlp_exp2_{cfg['name']}.pt")
    
    end_time = time.time()
    epoch_time = (end_time - start_time) / 10
    
    model.load_state_dict(torch.load(f"../checkpoints/mlp_exp2_{cfg['name']}.pt"))
    test_true, test_pred = get_predictions(model, test_loader, device, is_rnn=False)
    test_metrics = calculate_metrics(test_true, test_pred)
    
    result = {
        "Config": cfg['name'],
        "Accuracy": test_metrics['Accuracy'],
        "Precision": test_metrics['Precision'],
        "Recall": test_metrics['Recall'],
        "F1": test_metrics['F1 Score'],
        "Params": num_params,
        "Time/epoch": epoch_time
    }
    exp2_results.append(result)
    print(f"  Test Accuracy: {test_metrics['Accuracy']:.4f}")

exp2_df = pd.DataFrame(exp2_results)
print("\nExperiment 2 Results:")
print(exp2_df.to_string(index=False))



EXPERIMENT 2: Embedding Dimension (using best depth from Exp1)

Training: embed_64 (embed_dim=64, hidden_dims=[256])
  Epoch 1/10 | Train Loss: 0.5841 | Train Acc: 0.6833 | Val Acc: 0.7824
  Epoch 2/10 | Train Loss: 0.3786 | Train Acc: 0.8348 | Val Acc: 0.8408
  Epoch 3/10 | Train Loss: 0.2874 | Train Acc: 0.8819 | Val Acc: 0.8456
  Epoch 4/10 | Train Loss: 0.2297 | Train Acc: 0.9076 | Val Acc: 0.8632
  Epoch 5/10 | Train Loss: 0.1835 | Train Acc: 0.9301 | Val Acc: 0.8660
  Epoch 6/10 | Train Loss: 0.1455 | Train Acc: 0.9483 | Val Acc: 0.8664
  Epoch 7/10 | Train Loss: 0.1135 | Train Acc: 0.9617 | Val Acc: 0.8644
  Epoch 8/10 | Train Loss: 0.0861 | Train Acc: 0.9740 | Val Acc: 0.8680
  Epoch 9/10 | Train Loss: 0.0623 | Train Acc: 0.9835 | Val Acc: 0.8648
  Epoch 10/10 | Train Loss: 0.0445 | Train Acc: 0.9906 | Val Acc: 0.8612
  Test Accuracy: 0.8662

Training: embed_128 (embed_dim=128, hidden_dims=[256])
  Epoch 1/10 | Train Loss: 0.5400 | Train Acc: 0.7202 | Val Acc: 0.8160
  Epoch 2

In [15]:

# ============================================================================
# EXPERIMENT 3: Dropout Rate (0.2, 0.3, 0.5)
# Using best configs from Experiment 2
# ============================================================================
print("\n" + "="*70)
print("EXPERIMENT 3: Dropout Rate (using best embed_dim from Exp2)")
print("="*70)

best_embed_config = exp2_df.loc[exp2_df['Accuracy'].idxmax(), 'Config']
best_embed_dim = exp2_configs[[c['name'] for c in exp2_configs].index(best_embed_config)]['embed_dim']

exp3_configs = [
    {"name": "dropout_0.2", "dropout": 0.2, "embed_dim": best_embed_dim, "hidden_dims": best_hidden_dims},
    {"name": "dropout_0.3", "dropout": 0.3, "embed_dim": best_embed_dim, "hidden_dims": best_hidden_dims},
    {"name": "dropout_0.5", "dropout": 0.5, "embed_dim": best_embed_dim, "hidden_dims": best_hidden_dims},
]

exp3_results = []

for cfg in exp3_configs:
    print(f"\nTraining: {cfg['name']} (dropout={cfg['dropout']}, embed_dim={cfg['embed_dim']}, hidden_dims={cfg['hidden_dims']})")
    
    model = MLPClassifier(len(vocab), cfg['embed_dim'], cfg['hidden_dims'], 2, dropout=cfg['dropout']).to(device)
    num_params = count_parameters(model)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_acc = 0.0
    
    start_time = time.time()
    
    for epoch in range(10):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, is_rnn=False)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                val_loss += criterion(outputs, labels).item() * inputs.size(0)
        val_loss /= len(val_dataset)
        
        val_true, val_pred = get_predictions(model, val_loader, device, is_rnn=False)
        val_acc = calculate_metrics(val_true, val_pred)['Accuracy']
        
        print(f"  Epoch {epoch+1}/10 | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"../checkpoints/mlp_exp3_{cfg['name']}.pt")
    
    end_time = time.time()
    epoch_time = (end_time - start_time) / 10
    
    model.load_state_dict(torch.load(f"../checkpoints/mlp_exp3_{cfg['name']}.pt"))
    test_true, test_pred = get_predictions(model, test_loader, device, is_rnn=False)
    test_metrics = calculate_metrics(test_true, test_pred)
    
    result = {
        "Config": cfg['name'],
        "Accuracy": test_metrics['Accuracy'],
        "Precision": test_metrics['Precision'],
        "Recall": test_metrics['Recall'],
        "F1": test_metrics['F1 Score'],
        "Params": num_params,
        "Time/epoch": epoch_time
    }
    exp3_results.append(result)
    print(f"  Test Accuracy: {test_metrics['Accuracy']:.4f}")

exp3_df = pd.DataFrame(exp3_results)
print("\nExperiment 3 Results:")
print(exp3_df.to_string(index=False))



EXPERIMENT 3: Dropout Rate (using best embed_dim from Exp2)

Training: dropout_0.2 (dropout=0.2, embed_dim=128, hidden_dims=[256])
  Epoch 1/10 | Train Loss: 0.5304 | Train Acc: 0.7273 | Val Acc: 0.8204
  Epoch 2/10 | Train Loss: 0.3318 | Train Acc: 0.8561 | Val Acc: 0.8544
  Epoch 3/10 | Train Loss: 0.2471 | Train Acc: 0.9010 | Val Acc: 0.8660
  Epoch 4/10 | Train Loss: 0.1861 | Train Acc: 0.9307 | Val Acc: 0.8700
  Epoch 5/10 | Train Loss: 0.1401 | Train Acc: 0.9494 | Val Acc: 0.8632
  Epoch 6/10 | Train Loss: 0.0998 | Train Acc: 0.9675 | Val Acc: 0.8636
  Epoch 7/10 | Train Loss: 0.0684 | Train Acc: 0.9791 | Val Acc: 0.8640
  Epoch 8/10 | Train Loss: 0.0416 | Train Acc: 0.9901 | Val Acc: 0.8624
  Epoch 9/10 | Train Loss: 0.0231 | Train Acc: 0.9955 | Val Acc: 0.8612
  Epoch 10/10 | Train Loss: 0.0131 | Train Acc: 0.9987 | Val Acc: 0.8612
  Test Accuracy: 0.8626

Training: dropout_0.3 (dropout=0.3, embed_dim=128, hidden_dims=[256])
  Epoch 1/10 | Train Loss: 0.5271 | Train Acc: 0.729

In [16]:
# ============================================================================
# Save Best Overall MLP Model
# ============================================================================
print("\n" + "="*70)
print("SUMMARY: All Experiments Complete")
print("="*70)

all_results = pd.concat([exp1_df, exp2_df, exp3_df], ignore_index=True)
best_idx = all_results['Accuracy'].idxmax()
best_config = all_results.loc[best_idx]

print(f"\nBest MLP Configuration: {best_config['Config']}")
print(f"Test Accuracy: {best_config['Accuracy']:.4f}")
print(f"Precision: {best_config['Precision']:.4f}, Recall: {best_config['Recall']:.4f}, F1: {best_config['F1']:.4f}")
print(f"Time per epoch: {best_config['Time/epoch']:.2f}s")

# Save best model as mlp_best.pt
config_name = best_config['Config']

if "layer" in config_name:
    # Nếu tên config chứa chữ "layer" -> thuộc Experiment 1
    src = f"../checkpoints/mlp_exp1_{config_name.replace(' ', '_')}.pt"
elif "embed" in config_name:
    # Nếu tên config chứa chữ "embed" -> thuộc Experiment 2
    src = f"../checkpoints/mlp_exp2_{config_name}.pt"
else:
    # Còn lại (chứa chữ "dropout") -> thuộc Experiment 3
    src = f"../checkpoints/mlp_exp3_{config_name}.pt"

torch.save(torch.load(src), "../checkpoints/mlp_best.pt")
print(f"Best model saved from {src} to ../checkpoints/mlp_best.pt")
print(f"Best model saved to ../checkpoints/mlp_best.pt")

print("\n" + "="*70)
print("All Experiment Results Summary:")
print("="*70)
print(all_results.to_string(index=False))


SUMMARY: All Experiments Complete

Best MLP Configuration: embed_128
Test Accuracy: 0.8698
Precision: 0.8744, Recall: 0.8679, F1: 0.8711
Time per epoch: 1.41s
Best model saved from ../checkpoints/mlp_exp2_embed_128.pt to ../checkpoints/mlp_best.pt
Best model saved to ../checkpoints/mlp_best.pt

All Experiment Results Summary:
         Config  Accuracy  Precision   Recall       F1  Params  Time/epoch
 1 hidden layer    0.8640   0.853333 0.883629 0.868217 2593538    1.430641
2 hidden layers    0.8620   0.870929 0.854438 0.862605 2593282    1.544341
3 hidden layers    0.8634   0.856703 0.877318 0.866888 2585757    1.654453
       embed_64    0.8662   0.859676 0.879684 0.869565 1297154    1.230554
      embed_128    0.8698   0.874404 0.867850 0.871115 2593538    1.414104
      embed_256    0.8660   0.866982 0.869034 0.868006 5186306    2.101576
    dropout_0.2    0.8626   0.883721 0.839448 0.861016 2593538    1.454300
    dropout_0.3    0.8650   0.865279 0.869034 0.867152 2593538    1.459